# Génération de texte avec un RNN — corpus Victor Hugo

Ce troisième exercice traite des **données textuelles**. Nous allons apprendre à un réseau récurrent à prédire le mot suivant à partir des trois mots précédents, puis utiliser ces prédictions pour générer du texte mot après mot.

Le travail suivra le même cycle que les exercices précédents :

1. comprendre et nettoyer le corpus ;
2. séparer entraînement, validation et test sans fuite ;
3. construire le vocabulaire sur l’entraînement uniquement ;
4. fabriquer les séquences entrée-cible ;
5. comprendre puis construire un SimpleRNN ;
6. entraîner avec validation et arrêt anticipé ;
7. évaluer et générer du texte ;
8. comparer SimpleRNN et LSTM.

Chaque notion théorique et chaque formule seront présentées avant le code correspondant.

## 1. Définir précisément le problème

### 1.1 Prédire le mot suivant

Après nettoyage et tokenisation, un poème devient une suite d’identifiants de mots :

$$
(w_1,w_2,\ldots,w_T).
$$

Pour une position $t$, l’entrée contient au maximum les trois mots précédents :

$$
x_t=(w_{t-3},w_{t-2},w_{t-1}),
\qquad y_t=w_t.
$$

Au début d’un texte, lorsqu’il manque un ou plusieurs mots, nous ajouterons à gauche l’identifiant de bourrage 0 afin de conserver une entrée de longueur 3.

Le réseau devra produire une distribution de probabilités sur le vocabulaire :

$$
p_{\theta}(w_t=k\mid w_{t-3},w_{t-2},w_{t-1}),
\qquad k\in\{1,\ldots,V\},
$$

où $V$ est la taille du vocabulaire et $\theta$ représente tous les paramètres appris. Pour $N$ exemples, l’apprentissage cherchera plus tard à minimiser l’entropie croisée :

$$
L(\theta)=-\frac{1}{N}\sum_{n=1}^{N}
\log p_{\theta}(y_n\mid x_n).
$$

La génération répétera ensuite le même mécanisme : prédire un mot, l’ajouter au texte, déplacer la fenêtre de trois mots, puis prédire le suivant.

### 1.2 Protocole anti-fuite pour des données séquentielles

Deux fenêtres voisines se ressemblent presque entièrement. Par exemple :

$$
(w_1,w_2,w_3)\rightarrow w_4
$$

et

$$
(w_2,w_3,w_4)\rightarrow w_5
$$

partagent plusieurs mots. Si nous fabriquions toutes les fenêtres puis les répartissions au hasard, des fragments presque identiques pourraient se retrouver dans l’entraînement et le test. La performance semblerait alors meilleure qu’elle ne l’est réellement.

Notre protocole sera donc le suivant :

1. reconstruire les poèmes et préserver leurs frontières ;
2. séparer les **poèmes** en entraînement, validation et test ;
3. adapter le vocabulaire uniquement sur les poèmes d’entraînement ;
4. transformer ensuite chaque partie en fenêtres, sans jamais traverser la frontière entre deux poèmes ;
5. utiliser la validation pour les choix d’apprentissage ;
6. consulter le test une seule fois pour l’évaluation finale.

Cette étape est essentielle : le Tokenizer apprend lui aussi des informations sur les données, notamment quels mots existent et lesquels sont fréquents. Il fait donc partie du prétraitement à ajuster uniquement sur l’entraînement.

**Dans cette première étape, nous ne tokenisons encore rien.** Nous vérifions d’abord la qualité et la structure des fichiers.

## 2. Préparer l’environnement reproductible

Nous utilisons :

- *pathlib* pour manipuler les chemins de fichiers ;
- *re* pour quelques statistiques textuelles simples ;
- NumPy pour les futurs tableaux numériques ;
- TensorFlow/Keras pour le futur réseau récurrent.

Une graine fixe rend les séparations et initialisations reproductibles. Le déterminisme signifie que deux exécutions dans le même environnement doivent produire les mêmes résultats. Il ne garantit pas que deux versions différentes de TensorFlow donnent exactement les mêmes nombres.

In [1]:
from pathlib import Path
import re

import numpy as np
import tensorflow as tf

SEED = 42

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Graine     :", SEED)
print("Environnement prêt.")

TensorFlow : 2.21.0
NumPy      : 2.4.6
Graine     : 42
Environnement prêt.


## 3. Charger les fichiers sans transformer le texte

Le dossier contient deux fichiers :

- *victor_hugo-texts.txt* : le corpus concaténé ;
- *victor_hugo-titles.txt* : la liste des titres, utile pour retrouver les frontières entre les poèmes.

Un fichier texte contient d’abord des **octets**. Un encodage indique comment convertir ces octets en caractères Unicode. Nous demandons explicitement UTF-8 et une lecture stricte : si les octets n’étaient pas valides, Python lèverait une erreur au lieu de remplacer silencieusement des caractères.

À ce stade, nous conservons le contenu exactement tel qu’il est. Nous ne mettons pas le texte en minuscules, nous ne supprimons pas la ponctuation et nous ne construisons aucun vocabulaire.

In [2]:
DOSSIER_PROJET = Path.cwd()
CHEMIN_CORPUS = DOSSIER_PROJET / "victor_hugo-texts.txt"
CHEMIN_TITRES = DOSSIER_PROJET / "victor_hugo-titles.txt"

assert CHEMIN_CORPUS.is_file(), f"Fichier absent : {CHEMIN_CORPUS}"
assert CHEMIN_TITRES.is_file(), f"Fichier absent : {CHEMIN_TITRES}"

octets_corpus = CHEMIN_CORPUS.read_bytes()
octets_titres = CHEMIN_TITRES.read_bytes()

# errors="strict" est la valeur par défaut : aucune erreur n’est masquée.
texte_brut = octets_corpus.decode("utf-8", errors="strict")
texte_titres_brut = octets_titres.decode("utf-8", errors="strict")
titres_bruts = [
    ligne.strip()
    for ligne in texte_titres_brut.splitlines()
    if ligne.strip()
]

assert len(texte_brut) > 0
assert len(titres_bruts) > 0
assert "\ufffd" not in texte_brut
assert "\ufffd" not in texte_titres_brut

print("Dossier du projet :", DOSSIER_PROJET)
print("Corpus             :", CHEMIN_CORPUS.name, f"({len(octets_corpus):,} octets)")
print("Titres             :", CHEMIN_TITRES.name, f"({len(octets_titres):,} octets)")
print("Décodage UTF-8 strict réussi.")
print("Nombre de titres non vides :", len(titres_bruts))

Dossier du projet : C:\Users\driou\OneDrive\Bureau\introduction_au_deeplearning
Corpus             : victor_hugo-texts.txt (107,013 octets)
Titres             : victor_hugo-titles.txt (1,180 octets)
Décodage UTF-8 strict réussi.
Nombre de titres non vides : 48


## 4. Distinguer UTF-8 valide et texte correctement affiché

Un décodage UTF-8 réussi ne suffit pas à prouver que le texte est correct. Le contenu peut avoir été mal décodé dans le passé, puis réenregistré proprement en UTF-8. Cette corruption s’appelle le **mojibake**.

Par exemple, le mot attendu « siècle » peut devenir « siÃ¨cle » après cette chaîne d’opérations :

$$
\text{caractères corrects}
\longrightarrow
\text{octets UTF-8}
\longrightarrow
\text{mauvais décodage}
\longrightarrow
\text{caractères altérés}.
$$

Les suites contenant notamment « Ã », « Â » ou « â€ » sont des indices fréquents de ce problème en français. Nous allons les compter et afficher un extrait avant de décider d’une réparation. Modifier immédiatement le texte sans diagnostic pourrait détruire des caractères déjà corrects.

In [3]:
motifs_mojibake = {
    "Ã": texte_brut.count("Ã"),
    "Â": texte_brut.count("Â"),
    "â€": texte_brut.count("â€"),
}
nombre_indices_mojibake = sum(motifs_mojibake.values())

lignes_suspectes = [
    ligne for ligne in texte_brut.splitlines()
    if any(motif in ligne for motif in motifs_mojibake)
]

print("Indices de mojibake dans le corpus :")
for motif, nombre in motifs_mojibake.items():
    print(f"  {motif!r:<5} : {nombre:>5}")

print("\nPremier extrait suspect :")
print(lignes_suspectes[0] if lignes_suspectes else "Aucun extrait suspect")

assert nombre_indices_mojibake > 0
assert len(lignes_suspectes) > 0

print(
    "\nDiagnostic : le fichier est un UTF-8 valide, "
    "mais son contenu textuel doit être réparé avant la tokenisation."
)

Indices de mojibake dans le corpus :
  'Ã'   :  2124
  'Â'   :    90
  'â€'  :    94

Premier extrait suspect :
Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,

Diagnostic : le fichier est un UTF-8 valide, mais son contenu textuel doit être réparé avant la tokenisation.


## 5. Observer la structure générale du corpus

Avant de nettoyer, nous calculons seulement des statistiques descriptives : nombre de caractères, lignes, titres et unités séparées par des espaces.

Le comptage d’unités ci-dessous n’est **pas** encore la tokenisation du modèle. Il sert uniquement à estimer la taille du corpus. La vraie tokenisation déterminera plus tard précisément :

- la gestion des majuscules ;
- la ponctuation conservée ou supprimée ;
- le mot réservé aux éléments hors vocabulaire ;
- la taille maximale du vocabulaire.

Nous vérifions aussi combien de titres apparaissent littéralement dans le corpus. Cela nous indique si la liste des titres peut servir de repère pour reconstruire les poèmes.

In [4]:
lignes_corpus = texte_brut.splitlines()
lignes_non_vides = [ligne.strip() for ligne in lignes_corpus if ligne.strip()]
unites_approx = re.findall(r"\S+", texte_brut)
titres_retrouves = [titre for titre in titres_bruts if titre in texte_brut]

print("Caractères Unicode        :", f"{len(texte_brut):,}")
print("Lignes totales            :", f"{len(lignes_corpus):,}")
print("Lignes non vides          :", f"{len(lignes_non_vides):,}")
print("Unités séparées par espace:", f"{len(unites_approx):,}")
print("Titres fournis            :", len(titres_bruts))
print("Titres retrouvés tels quels:", len(titres_retrouves))

print("\nDix premiers titres bruts :")
for indice, titre in enumerate(titres_bruts[:10], start=1):
    print(f"{indice:>2}. {titre}")

print("\nDébut brut du corpus :")
print("\n".join(lignes_corpus[:12]))

assert len(titres_retrouves) > 0
assert len(unites_approx) > len(titres_bruts)

Caractères Unicode        : 102,257
Lignes totales            : 2,349
Lignes non vides          : 1,400
Unités séparées par espace: 17,309
Titres fournis            : 48
Titres retrouvés tels quels: 13

Dix premiers titres bruts :
 1. Mes vers fuiraient
 2. Ce siÃ¨cle avait deux ans
 3. Le mariage de Roland
 4. Lâ€™enfant
 5. La ronde du sabbat
 6. Un jour je vis
 7. Melancholia
 8. Lâ€™hiver
 9. Jâ€™eus toujours de lâ€™amour
10. Chanson

Début brut du corpus :
Mes vers fuiraient

Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,
Si mes vers avaient des ailes,
Des ailes comme l'oiseau.


Ils voleraient, Ã©tincelles, Vers votre foyer qui rit,
Si mes vers avaient des ailes,
Des ailes comme l'esprit.

PrÃ¨s de vous, purs et fidÃ¨les, Ils accourraient, nuit et jour, Si mes vers avaient des ailes, Des ailes comme l'amour!


## Bilan de l’étape 1

Le problème est maintenant défini : prédire chaque mot à partir d’une fenêtre maximale de trois mots, puis réutiliser le modèle de façon autorégressive pour générer du texte.

Le corpus brut contient :

- **102 257 caractères Unicode** ;
- **2 349 lignes**, dont 1 400 non vides ;
- environ **17 309 unités** séparées par des espaces ;
- **48 titres** fournis séparément.

Le diagnostic d’encodage révèle **2 124 occurrences de « Ã »**, 90 de « Â » et 94 de « â€ ». Le fichier est bien décodable en UTF-8, mais une corruption antérieure a transformé des mots comme « frêles » en « frÃªles ». Tokeniser maintenant créerait artificiellement des mots incorrects et polluerait le vocabulaire.

Seulement 13 titres sur 48 sont retrouvés littéralement dans le corpus brut. Nous ne devons donc pas découper naïvement le texte à partir des titres avant d’avoir réparé et normalisé les deux fichiers de la même manière.

Enfin, le protocole anti-fuite est fixé : la future séparation se fera **par poème avant la création des fenêtres**, et le vocabulaire sera appris uniquement sur les poèmes d’entraînement.

**Prochaine étape :** réparer le mojibake de manière contrôlée, vérifier la correction sur plusieurs exemples, puis reconstruire les 48 poèmes sans encore créer le vocabulaire.

## 6. Réparer le mojibake sans modifier les fichiers sources

### 6.1 Comprendre l’opération inverse

Le texte est bien stocké aujourd’hui dans un fichier UTF-8, mais son contenu avait déjà été mal interprété auparavant. Notons $E_e$ l’encodage en octets avec l’encodage $e$, et $D_e$ le décodage de ces octets. Pour un texte correct $t$, le mojibake observé $m$ provient généralement de :

$$
m = D_{\mathrm{CP1252}}\left(E_{\mathrm{UTF8}}(t)\right).
$$

Si aucune information n’a été perdue, l’opération inverse est :

$$
t = D_{\mathrm{UTF8}}\left(E_{\mathrm{CP1252}}(m)\right).
$$

Dans le corpus, cette inversion révèle exactement **255 séquences invalides**. Elles ont toutes la même forme : l’octet `A0` du caractère « à », normalement encodé `C3 A0` en UTF-8, a été remplacé par un espace `20`. Nous autorisons donc une seule réparation binaire connue :

$$
\mathtt{C3\ 20} \longrightarrow \mathtt{C3\ A0}.
$$

Cette correction consomme l’espace qui remplace l’octet perdu ; elle n’ajoute donc pas d’espace artificiel. Toute autre erreur provoquera volontairement une exception. Les variables brutes et les deux fichiers restent intacts.


In [5]:
import hashlib
import unicodedata


def trouver_positions_invalides_utf8(octets):
    """Retourne les positions qui empêchent un décodage UTF-8 strict."""
    positions = []
    position = 0

    while position < len(octets):
        try:
            octets[position:].decode("utf-8", errors="strict")
            break
        except UnicodeDecodeError as erreur:
            position_erreur = position + erreur.start
            positions.append(position_erreur)
            position = position_erreur + 1

    return positions


def reparer_mojibake_cp1252(texte, autoriser_a_endommage=False):
    """Inverse le mauvais décodage CP1252, sans ignorer aucune erreur."""
    octets_reconstitues = texte.encode("cp1252", errors="strict")
    positions_invalides = trouver_positions_invalides_utf8(octets_reconstitues)

    if positions_invalides:
        if not autoriser_a_endommage:
            raise ValueError("Séquence UTF-8 endommagée non autorisée.")

        motifs_invalides = {
            octets_reconstitues[position:position + 2]
            for position in positions_invalides
        }
        if motifs_invalides != {b"\xc3 "}:
            raise ValueError(f"Motif inattendu : {motifs_invalides}")

        # C3 20 devient C3 A0 : le caractère « à » est restauré.
        octets_reconstitues = octets_reconstitues.replace(
            b"\xc3 ",
            b"\xc3\xa0",
        )

    texte_repare = octets_reconstitues.decode("utf-8", errors="strict")
    texte_repare = unicodedata.normalize("NFC", texte_repare)
    return texte_repare, positions_invalides


empreinte_corpus_source = hashlib.sha256(octets_corpus).hexdigest()
empreinte_titres_source = hashlib.sha256(octets_titres).hexdigest()

texte_corrige, positions_a_reparees = reparer_mojibake_cp1252(
    texte_brut,
    autoriser_a_endommage=True,
)
texte_titres_corrige, erreurs_titres = reparer_mojibake_cp1252(
    texte_titres_brut
)
titres_corriges = [
    ligne.strip()
    for ligne in texte_titres_corrige.splitlines()
    if ligne.strip()
]

print("Séquences « à » réparées dans le corpus :", len(positions_a_reparees))
print("Erreurs binaires dans le fichier des titres :", len(erreurs_titres))
print("Caractères après réparation :", f"{len(texte_corrige):,}")


Séquences « à » réparées dans le corpus : 255
Erreurs binaires dans le fichier des titres : 0
Caractères après réparation : 99,835


### 6.2 Valider la réparation avant toute analyse

Une sortie lisible à l’œil ne suffit pas. Nous exigeons simultanément :

- le même nombre de lignes avant et après ;
- exactement 48 titres, tous différents ;
- aucun caractère de remplacement `�` et aucun motif de mojibake connu ;
- plusieurs mots témoins correctement restaurés ;
- les mêmes empreintes SHA-256 pour les fichiers sur disque, preuve qu’ils n’ont pas été réécrits.

En notation compacte, si $L$ compte les lignes et $T$ les titres, nous vérifions :

$$
L_{\mathrm{brut}} = L_{\mathrm{corrigé}} = 2349,
\qquad T_{\mathrm{corrigé}} = 48.
$$

Attention : nous ne demandons pas la disparition de toute lettre « Â ». Le corpus contient réellement la phrase « Âme où rien n’est impur ». Un bon contrôle distingue un caractère français légitime d’un motif corrompu comme `Â«`.


In [6]:
motifs_mojibake_residuels = ("Ã", "â€", "Å“", "Â«", "Â»", "\ufffd")

assert len(positions_a_reparees) == 255
assert len(erreurs_titres) == 0
assert len(texte_corrige.splitlines()) == len(texte_brut.splitlines()) == 2349
assert len(titres_corriges) == len(titres_bruts) == 48
assert len(set(titres_corriges)) == 48

for motif in motifs_mojibake_residuels:
    assert motif not in texte_corrige, f"Motif résiduel : {motif!r}"
    assert motif not in texte_titres_corrige, f"Motif résiduel dans les titres : {motif!r}"

# Le seul « Â » restant est une vraie majuscule accentuée.
assert texte_corrige.count("Â") == 1
assert "Âme où rien n'est impur" in texte_corrige
assert "Â" not in texte_titres_corrige

temoins = [
    ("frÃªles", "frêles"),
    ("Ã©tincelles", "étincelles"),
    ("siÃ¨cle", "siècle"),
    ("cÅ“ur", "cœur"),
    ("Lâ€™enfant", "L’enfant"),
]
for forme_abimee, forme_corrigee in temoins:
    assert forme_abimee in (texte_brut + texte_titres_brut)
    assert forme_corrigee in (texte_corrige + texte_titres_corrige)

# Relecture des fichiers : leur contenu binaire doit être inchangé.
assert hashlib.sha256(CHEMIN_CORPUS.read_bytes()).hexdigest() == empreinte_corpus_source
assert hashlib.sha256(CHEMIN_TITRES.read_bytes()).hexdigest() == empreinte_titres_source

print("Réparation validée sans modification des fichiers sources.")
print("Lignes du corpus :", len(texte_corrige.splitlines()))
print("Titres uniques   :", len(set(titres_corriges)))
print()
print("Avant :")
print("\n".join(texte_brut.splitlines()[:4]))
print()
print("Après :")
print("\n".join(texte_corrige.splitlines()[:4]))


Réparation validée sans modification des fichiers sources.
Lignes du corpus : 2349
Titres uniques   : 48

Avant :
Mes vers fuiraient

Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,
Si mes vers avaient des ailes,

Après :
Mes vers fuiraient

Mes vers fuiraient, doux et frêles, Vers votre jardin si beau,
Si mes vers avaient des ailes,


## 7. Reconstruire les 48 poèmes

### 7.1 Pourquoi une découpe automatique serait trompeuse

Trois séparateurs apparemment naturels ont été vérifiés puis écartés :

- les 48 titres ne figurent pas tous littéralement dans le corpus ;
- les suites de lignes vides séparent aussi des strophes ;
- les 47 mentions « Victor Hugo » ne représentent pas 47 fins fiables.

Deux anomalies sont particulièrement importantes. Une attribution coupe **Les Djinns** en plein milieu, et une autre coupe **Oceano Nox**. Inversement, aucune attribution ne sépare **La chasse du Burgave** de **Jeanne était au pain sec**.

Pour ce fichier fixe, nous utilisons donc un **manifeste relu** de 48 intervalles. Une borne $(d_i,f_i)$ est semi-ouverte : elle conserve les indices de ligne $j$ tels que

$$
d_i \leq j < f_i.
$$

Ce choix évite les erreurs silencieuses. Les quatre lignes bibliographiques insérées à l’intérieur de deux poèmes sont nommées et retirées explicitement. Ce découpage ne crée encore ni jetons, ni identifiants numériques, ni vocabulaire.


In [7]:
lignes_corrigees = texte_corrige.splitlines()

# Indices Python : début inclus, fin exclue.
BORNES_POEMES = [
    (0, 13),       #  1. Mes vers fuiraient
    (15, 69),      #  2. Ce siècle avait deux ans
    (74, 149),     #  3. Le mariage de Roland
    (153, 159),    #  4. L’enfant
    (168, 263),    #  5. La ronde du sabbat
    (281, 301),    #  6. Un jour je vis
    (311, 344),    #  7. Melancholia
    (351, 365),    #  8. L’hiver
    (370, 380),    #  9. J’eus toujours de l’amour
    (390, 410),    # 10. Chanson
    (417, 588),    # 11. La chasse du Burgave
    (588, 605),    # 12. Jeanne était au pain sec
    (611, 652),    # 13. Lorsque l’enfant paraît...
    (665, 680),    # 14. Ordre du jour de floréal
    (688, 726),    # 15. Spectacle rassurant
    (735, 770),    # 16. À l’obéissance passive (I)
    (778, 969),    # 17. Fonction du poète
    (983, 998),    # 18. Fable ou histoire
    (1008, 1034),  # 19. Regardez : les enfants sont assis en rond
    (1041, 1081),  # 20. J’aime l’araignée et j’aime l’ortie
    (1086, 1224),  # 21. Les Djinns
    (1224, 1245),  # 22. Printemps
    (1252, 1262),  # 23. Unité
    (1268, 1299),  # 24. Les raisons du Momotombo
    (1303, 1321),  # 25. Sur une barricade...
    (1334, 1346),  # 26. La lune
    (1350, 1373),  # 27. Clair de lune
    (1380, 1410),  # 28. Vieille chanson du jeune temps
    (1416, 1440),  # 29. La coccinelle
    (1453, 1489),  # 30. Jeanne endormie
    (1500, 1526),  # 31. La pauvre fleur disait au papillon céleste
    (1535, 1731),  # 32. Les pauvres gens
    (1746, 1760),  # 33. À une femme
    (1766, 1828),  # 34. Guitare
    (1842, 1850),  # 35. Mes deux filles
    (1855, 1867),  # 36. Viens ! – une flûte invisible
    (1873, 1900),  # 37. Chanson de grand-père
    (1904, 1917),  # 38. La vache
    (1931, 1947),  # 39. Demain, dès l’aube
    (1950, 1987),  # 40. Je suis des bois l’hôte fidèle
    (1994, 2111),  # 41. Tristesse d’Olympio
    (2119, 2185),  # 42. Ce que dit le public
    (2191, 2209),  # 43. La source tombait du rocher
    (2221, 2234),  # 44. Le poème du jardin des plantes
    (2240, 2259),  # 45. Les enfants gâtés
    (2272, 2283),  # 46. Après la bataille
    (2288, 2331),  # 47. Oceano Nox
    (2331, 2346),  # 48. Ceux qui vivent, ce sont ceux qui luttent
]

# Métadonnées placées à tort au milieu d’un poème (indices 0-based).
METADONNEES_INTERNES = {
    1153: "Victor Hugo:",
    1158: "Les Orientales",
    1159: "1829",
    2304: "Victor Hugo: Les Rayons et les ombres juillet 1836",
}

assert len(BORNES_POEMES) == len(titres_corriges) == 48
for indice, valeur_attendue in METADONNEES_INTERNES.items():
    assert lignes_corrigees[indice].strip() == valeur_attendue

poemes = []
indices_des_plages = set()
indices_de_contenu = set()

for numero, (titre, bornes) in enumerate(
    zip(titres_corriges, BORNES_POEMES),
    start=1,
):
    debut, fin = bornes
    assert 0 <= debut < fin <= len(lignes_corrigees)

    indices_courants = set(range(debut, fin))
    assert indices_des_plages.isdisjoint(indices_courants)
    indices_des_plages.update(indices_courants)

    indices_utiles = [
        indice
        for indice in range(debut, fin)
        if indice not in METADONNEES_INTERNES
    ]
    indices_de_contenu.update(indices_utiles)
    texte_poeme = "\n".join(
        lignes_corrigees[indice] for indice in indices_utiles
    ).strip()

    assert texte_poeme, f"Poème vide : {titre}"
    poemes.append({
        "numero": numero,
        "titre": titre,
        "texte": texte_poeme,
        "ligne_debut": debut + 1,
        "ligne_fin": fin,
    })


### 7.2 Vérifier que la reconstruction est complète et cohérente

Le manifeste n’est accepté que si les 48 plages sont ordonnées, disjointes et non vides. Nous contrôlons aussi plusieurs ancres situées aux endroits difficiles, notamment juste avant et après les séparations manquantes ou déplacées.

Enfin, chaque ligne non vide reçoit exactement l’une de ces deux étiquettes :

$$
\mathcal{L}_{\mathrm{non\ vide}}
= \mathcal{L}_{\mathrm{poèmes}}
\,\dot\cup\,
\mathcal{L}_{\mathrm{métadonnées}},
$$

où $\dot\cup$ indique une union sans chevauchement. Ici, 1 287 lignes non vides appartiennent aux poèmes et 113 sont des titres de source, attributions, collections ou dates volontairement exclues. Nous validons ainsi la structure, pas la qualité littéraire ni le futur vocabulaire.


In [8]:
assert len(poemes) == 48
assert [poeme["titre"] for poeme in poemes] == titres_corriges
assert all(poeme["texte"] for poeme in poemes)
assert all(
    BORNES_POEMES[indice][1] <= BORNES_POEMES[indice + 1][0]
    for indice in range(len(BORNES_POEMES) - 1)
)
assert all("Victor Hugo" not in poeme["texte"] for poeme in poemes)

ancres_debut = {
    1: "Mes vers fuiraient",
    5: "À M. Charles Nodier",
    12: "Jeanne était au pain sec",
    21: "Murs, ville",
    22: "C'est la jeunesse et le matin",
    47: "Oh ! combien de marins",
    48: "Ceux qui vivent, ce sont ceux qui luttent",
}
for numero, ancre in ancres_debut.items():
    assert poemes[numero - 1]["texte"].startswith(ancre)

indices_non_vides = {
    indice
    for indice, ligne in enumerate(lignes_corrigees)
    if ligne.strip()
}
indices_poemes_non_vides = indices_de_contenu & indices_non_vides
indices_metadonnees_non_vides = indices_non_vides - indices_poemes_non_vides

assert indices_poemes_non_vides.isdisjoint(indices_metadonnees_non_vides)
assert indices_poemes_non_vides | indices_metadonnees_non_vides == indices_non_vides
assert len(indices_poemes_non_vides) == 1287
assert len(indices_metadonnees_non_vides) == 113

print("Nombre exact de poèmes reconstruits :", len(poemes))
print("Lignes non vides de poésie          :", len(indices_poemes_non_vides))
print("Lignes bibliographiques exclues     :", len(indices_metadonnees_non_vides))

for numero in (1, 12, 21, 47, 48):
    poeme = poemes[numero - 1]
    premiere_ligne = poeme["texte"].splitlines()[0]
    print(f"{numero:>2}. {poeme['titre']} -> {premiere_ligne[:70]}")


Nombre exact de poèmes reconstruits : 48
Lignes non vides de poésie          : 1287
Lignes bibliographiques exclues     : 113
 1. Mes vers fuiraient -> Mes vers fuiraient
12. Jeanne était au pain sec -> Jeanne était au pain sec dans le cabinet noir,
21. Les Djinns -> Murs, ville Et port, Asile
47. Oceano Nox -> Oh ! combien de marins, combien de capitaines Qui sont partis joyeux p
48. Ceux qui vivent , ce sont ceux qui luttent -> Ceux qui vivent, ce sont ceux qui luttent ; ce sont Ceux dont un desse


## Bilan de l’étape 2

La réparation a été effectuée uniquement dans de nouvelles variables Python. Les deux fichiers sources et les variables brutes ont été conservés. Les 255 séquences endommagées correspondant à « à » ont été identifiées avant correction ; aucune autre erreur binaire n’a été tolérée. Les motifs de mojibake ont disparu, tandis que le vrai mot « Âme » a été préservé.

Le corpus a ensuite été reconstruit en **48 poèmes non vides**, associés dans l’ordre aux **48 titres uniques**. Le manifeste explicite documente les frontières et traite les deux attributions placées au milieu de « Les Djinns » et « Oceano Nox », ainsi que la séparation absente avant « Jeanne était au pain sec ».

À ce stade, la variable `poemes` contient seulement des dictionnaires avec un numéro, un titre, un texte et ses lignes d’origine. **Aucun tokenizer et aucun vocabulaire n’ont été créés.**

**Prochaine étape :** séparer les 48 poèmes en entraînement, validation et test sans fuite de données, puis seulement construire le vocabulaire à partir de l’entraînement.


## 8. Séparer les poèmes sans fuite de données

### 8.1 L’unité de séparation doit être le poème

Deux passages d’un même poème partagent le vocabulaire, les personnages, le rythme et parfois des vers répétés. Placer des fenêtres voisines dans deux jeux différents donnerait indirectement au modèle une partie du test pendant l’entraînement. Nous séparons donc les **48 poèmes entiers avant de créer la moindre fenêtre**.

Si $\mathcal{P}$ désigne l’ensemble des poèmes, nous voulons trois sous-ensembles disjoints :

$$
\mathcal{P}_{\mathrm{train}} \cap \mathcal{P}_{\mathrm{val}}
= \mathcal{P}_{\mathrm{train}} \cap \mathcal{P}_{\mathrm{test}}
= \mathcal{P}_{\mathrm{val}} \cap \mathcal{P}_{\mathrm{test}}
= \varnothing,
$$

$$
\mathcal{P}_{\mathrm{train}} \cup \mathcal{P}_{\mathrm{val}}
\cup \mathcal{P}_{\mathrm{test}} = \mathcal{P}.
$$

Avec 48 poèmes, nous choisissons 34 poèmes d’entraînement, 7 de validation et 7 de test. Un simple tirage peut être très déséquilibré, car certains poèmes sont beaucoup plus longs que d’autres. Nous classons donc les poèmes par longueur, divisons ce classement en sept tranches, puis tirons dans chaque tranche un poème pour le test et un pour la validation. Les autres vont dans l’entraînement.

Cette stratification n’utilise que le **nombre de caractères**, jamais les mots, le vocabulaire ou les performances du modèle. La graine fixe rend la séparation reproductible.


In [9]:
longueurs_caracteres = np.array([
    len(poeme["texte"]) for poeme in poemes
])
indices_par_longueur = np.argsort(
    -longueurs_caracteres,
    kind="stable",
)
tranches_de_longueur = np.array_split(indices_par_longueur, 7)

generateur_split = np.random.default_rng(SEED)
indices_entrainement = []
indices_validation = []
indices_test = []

for tranche in tranches_de_longueur:
    tranche_melangee = generateur_split.permutation(tranche)

    # Chaque niveau de longueur contribue aux trois jeux.
    indices_test.append(int(tranche_melangee[0]))
    indices_validation.append(int(tranche_melangee[1]))
    indices_entrainement.extend(map(int, tranche_melangee[2:]))

# L’ordre original facilite la lecture et ne change pas l’apprentissage.
indices_entrainement.sort()
indices_validation.sort()
indices_test.sort()

poemes_entrainement = [poemes[i] for i in indices_entrainement]
poemes_validation = [poemes[i] for i in indices_validation]
poemes_test = [poemes[i] for i in indices_test]

textes_entrainement = [p["texte"] for p in poemes_entrainement]
textes_validation = [p["texte"] for p in poemes_validation]
textes_test = [p["texte"] for p in poemes_test]


### 8.2 Auditer et figer la séparation

Une graine fixe ne suffit pas à prouver qu’une séparation est correcte. Nous contrôlons les tailles, l’absence de chevauchement, la présence de chacun des 48 identifiants exactement une fois et l’absence de textes dupliqués. Nous figeons aussi les numéros attendus : si le corpus ou l’algorithme change accidentellement, une assertion échouera au lieu de déplacer silencieusement un poème vers le test.

Le nombre de poèmes et le volume de texte sont deux mesures différentes. Pour un jeu $S$, la part de caractères est :

$$
r_S = \frac{\sum_{p \in S} \operatorname{len}(p)}
{\sum_{p \in \mathcal{P}} \operatorname{len}(p)}.
$$

La validation servira aux choix d’apprentissage et à l’arrêt anticipé. Le test est désormais mis en quarantaine : ses titres et sa taille documentent la séparation, mais aucun apprentissage ni choix de modèle ne s’appuiera sur son texte.


In [10]:
attendus_entrainement = [
    1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
    19, 22, 23, 24, 25, 27, 29, 31, 32, 36, 37, 40, 42, 45, 46, 47, 48,
]
attendus_validation = [20, 26, 30, 33, 34, 39, 41]
attendus_test = [3, 21, 28, 35, 38, 43, 44]

assert [i + 1 for i in indices_entrainement] == attendus_entrainement
assert [i + 1 for i in indices_validation] == attendus_validation
assert [i + 1 for i in indices_test] == attendus_test

ensemble_train = set(indices_entrainement)
ensemble_validation = set(indices_validation)
ensemble_test = set(indices_test)

assert len(ensemble_train) == 34
assert len(ensemble_validation) == 7
assert len(ensemble_test) == 7
assert ensemble_train.isdisjoint(ensemble_validation)
assert ensemble_train.isdisjoint(ensemble_test)
assert ensemble_validation.isdisjoint(ensemble_test)
assert ensemble_train | ensemble_validation | ensemble_test == set(range(48))

empreintes_poemes = {
    hashlib.sha256(poeme["texte"].encode("utf-8")).hexdigest()
    for poeme in poemes
}
assert len(empreintes_poemes) == 48

def statistiques_jeu(liste_poemes):
    return {
        "poemes": len(liste_poemes),
        "caracteres": sum(len(p["texte"]) for p in liste_poemes),
        "unites_approx": sum(len(p["texte"].split()) for p in liste_poemes),
    }

statistiques_splits = {
    "Entraînement": statistiques_jeu(poemes_entrainement),
    "Validation": statistiques_jeu(poemes_validation),
    "Test": statistiques_jeu(poemes_test),
}
total_caracteres = sum(s["caracteres"] for s in statistiques_splits.values())
assert total_caracteres == sum(len(p["texte"]) for p in poemes) == 94645
assert [s["caracteres"] for s in statistiques_splits.values()] == [
    66141, 14687, 13817
]

print(f"{'Jeu':<15} {'Poèmes':>7} {'Caractères':>12} {'Part':>9} {'Unités ≈':>10}")
for nom, stats in statistiques_splits.items():
    part = 100 * stats["caracteres"] / total_caracteres
    print(
        f"{nom:<15} {stats['poemes']:>7} {stats['caracteres']:>12,} "
        f"{part:>8.2f}% {stats['unites_approx']:>10,}"
    )

print()
print("Poèmes de validation :")
for poeme in poemes_validation:
    print(f"  {poeme['numero']:>2}. {poeme['titre']}")

print()
print("Poèmes de test scellés :")
for poeme in poemes_test:
    print(f"  {poeme['numero']:>2}. {poeme['titre']}")


Jeu              Poèmes   Caractères      Part   Unités ≈
Entraînement         34       66,141    69.88%     11,841
Validation            7       14,687    15.52%      2,653
Test                  7       13,817    14.60%      2,423

Poèmes de validation :
  20. J’aime l’araignée et j’aime l’ortie
  26. La lune
  30. Jeanne endormie
  33. À une femme
  34. Guitare
  39. Demain, dès l’aube
  41. Tristesse d’Olympio

Poèmes de test scellés :
   3. Le mariage de Roland
  21. Les Djinns
  28. Vieille chanson du jeune temps
  35. Mes deux filles
  38. La vache
  43. La source tombait du rocher
  44. Le poème du jardin des plantes


## 9. Construire le vocabulaire sur l’entraînement uniquement

### 9.1 De la chaîne de caractères aux mots

Un réseau de neurones ne reçoit pas directement les mots : chaque mot doit être associé à un entier. Avant cette association, une règle déterministe standardise le texte :

1. passage en minuscules ;
2. remplacement des apostrophes typographiques par `'` ;
3. conservation des lettres françaises et des apostrophes internes ;
4. remplacement de la ponctuation, des chiffres et des retours à la ligne par des espaces ;
5. réduction des espaces successifs.

Ainsi, « L’enfant rêve : déjà, grand-père ! » devient `l'enfant rêve déjà grand père`. Nous choisissons volontairement ici un premier modèle **au niveau des mots** : la ponctuation ne sera pas prédite. Cette simplification devra être rappelée lors de l’interprétation du texte généré.

Pour chaque mot $w$, sa fréquence d’entraînement est :

$$
c_{\mathrm{train}}(w)
= \sum_{p \in \mathcal{P}_{\mathrm{train}}}
\sum_{j=1}^{n_p} \mathbf{1}[u_{p,j}=w].
$$

Le vocabulaire contient uniquement les mots tels que $c_{\mathrm{train}}(w)>0$. `TextVectorization`, l’outil actuel de Keras pour ce travail, réserve aussi :

- l’identifiant 0 au remplissage, noté `<PAD>` dans nos explications ;
- l’identifiant 1 à `[UNK]`, pour un mot inconnu absent de l’entraînement.

L’appel `adapt()` est l’étape qui apprend le vocabulaire. Il recevra **seulement les 34 textes d’entraînement**. La validation peut ensuite être transformée sans modifier ce vocabulaire ; le test ne sera pas encore vectorisé.


In [11]:
CARACTERES_FRANCAIS = "a-zàâäæçèéêëîïñôöœùûüÿ"

def standardiser_francais(texte):
    texte = tf.strings.lower(texte, encoding="utf-8")
    texte = tf.strings.regex_replace(texte, "[’‘`´]", "'")

    motif_a_retirer = f"[^{CARACTERES_FRANCAIS}']+"
    texte = tf.strings.regex_replace(texte, motif_a_retirer, " " )
    texte = tf.strings.regex_replace(texte, r"\s+", " " )
    return tf.strings.strip(texte)


vectoriseur = tf.keras.layers.TextVectorization(
    standardize=standardiser_francais,
    split="whitespace",
    output_mode="int",
    ragged=True,
    name="vectoriseur_mots",
)

dataset_vocabulaire = tf.data.Dataset.from_tensor_slices(
    textes_entrainement
).batch(8)

# Seule opération apprenante de cette cellule : entraînement uniquement.
vectoriseur.adapt(dataset_vocabulaire)

vocabulaire = [str(mot) for mot in vectoriseur.get_vocabulary()]
taille_vocabulaire = vectoriseur.vocabulary_size()
ID_PADDING = 0
ID_HORS_VOCABULAIRE = 1
mot_vers_id = {mot: indice for indice, mot in enumerate(vocabulaire)}
id_vers_mot = np.array(vocabulaire)


### 9.2 Vérifier le vocabulaire et mesurer les mots inconnus

Nous reconstruisons indépendamment l’ensemble des mots standardisés de l’entraînement et vérifions qu’il correspond exactement aux entrées apprises, hors les deux identifiants réservés. Cela prouve notamment qu’aucun mot de validation ou de test n’a été ajouté par erreur.

Un mot de validation absent de l’entraînement devient `[UNK]`, d’identifiant 1. Pour $N$ identifiants de validation, le taux de mots hors vocabulaire est :

$$
r_{\mathrm{OOV}}
= \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}[z_i=1].
$$

Un taux non nul n’est pas une erreur : il mesure honnêtement la difficulté d’appliquer un vocabulaire appris sur un petit corpus à des poèmes jamais vus. Nous calculons ce diagnostic sur l’entraînement et la validation seulement. Le test reste scellé.


In [12]:
mots_train_controle = set()
for texte in textes_entrainement:
    texte_standardise = standardiser_francais(
        tf.constant(texte)
    ).numpy().decode("utf-8")
    mots_train_controle.update(texte_standardise.split())

assert vocabulaire[ID_PADDING] == ""
assert vocabulaire[ID_HORS_VOCABULAIRE] == "[UNK]"
assert taille_vocabulaire == len(vocabulaire)
assert set(vocabulaire[2:]) == mots_train_controle

vocabulaire_avant_validation = list(vocabulaire)
ids_entrainement = vectoriseur(tf.constant(textes_entrainement))
ids_validation = vectoriseur(tf.constant(textes_validation))
assert vectoriseur.get_vocabulary() == vocabulaire_avant_validation

valeurs_train = ids_entrainement.flat_values.numpy()
valeurs_validation = ids_validation.flat_values.numpy()
nombre_oov_train = int(np.count_nonzero(
    valeurs_train == ID_HORS_VOCABULAIRE
))
nombre_oov_validation = int(np.count_nonzero(
    valeurs_validation == ID_HORS_VOCABULAIRE
))
taux_oov_validation = 100 * nombre_oov_validation / len(valeurs_validation)

assert taille_vocabulaire == 3203
assert len(valeurs_train) == 11242
assert len(valeurs_validation) == 2529
assert nombre_oov_train == 0
assert nombre_oov_validation == 488

ids_mot_invente = vectoriseur(
    tf.constant(["zxqvbnmjamaisvu"])
).flat_values.numpy().tolist()
assert ids_mot_invente == [ID_HORS_VOCABULAIRE]

exemple = "L’enfant rêve : déjà, grand-père ! Cœur d’or."
exemple_standardise = standardiser_francais(
    tf.constant(exemple)
).numpy().decode("utf-8")
assert exemple_standardise == "l'enfant rêve déjà grand père cœur d'or"

print("Exemple original     :", exemple)
print("Exemple standardisé :", exemple_standardise)
print()
print("Taille totale du vocabulaire :", taille_vocabulaire)
print("Mots appris                  :", taille_vocabulaire - 2)
print("20 mots les plus fréquents  :")
print(vocabulaire[2:22])
print()
print("OOV entraînement :", nombre_oov_train, "sur", len(valeurs_train))
print(
    "OOV validation   :",
    nombre_oov_validation,
    "sur",
    len(valeurs_validation),
    f"({taux_oov_validation:.2f} %)",
)
print("Le jeu de test n’a pas été vectorisé.")


Exemple original     : L’enfant rêve : déjà, grand-père ! Cœur d’or.
Exemple standardisé : l'enfant rêve déjà grand père cœur d'or

Taille totale du vocabulaire : 3203
Mots appris                  : 3201
20 mots les plus fréquents  :
['et', 'les', 'le', 'la', 'de', 'dans', 'qui', 'à', 'des', 'que', 'un', 'au', 'en', 'sur', 'ce', 'il', 'du', 'comme', 'je', 'est']

OOV entraînement : 0 sur 11242
OOV validation   : 488 sur 2529 (19.30 %)
Le jeu de test n’a pas été vectorisé.


## Bilan de l’étape 3

Les 48 poèmes ont été répartis de façon reproductible et sans chevauchement : **34 pour l’entraînement, 7 pour la validation et 7 pour le test**. La stratification par longueur produit respectivement **69,88 %**, **15,52 %** et **14,60 %** des caractères. Aucun poème n’a été découpé entre deux jeux.

Le vocabulaire a ensuite été appris exclusivement sur les textes d’entraînement. Il contient **3 203 identifiants** : le remplissage, `[UNK]` et 3 201 mots observés. Aucun mot d’entraînement n’est inconnu, tandis que la validation contient **488 mots inconnus sur 2 529**, soit **19,30 %**. Ce taux élevé est plausible pour seulement 34 poèmes et devra être pris en compte dans l’interprétation des performances. Nous ne modifions pas le vocabulaire pour l’améliorer à partir de la validation.

Le jeu de test est toujours scellé : il n’a été ni utilisé par `adapt()`, ni vectorisé, ni évalué. Aucune fenêtre entrée-cible n’a encore été créée.

**Prochaine étape :** vectoriser chaque poème séparément, puis construire les exemples « trois mots précédents $\rightarrow$ mot suivant » sans qu’aucune fenêtre ne traverse une frontière de poème.


## 10. Vectoriser chaque poème séparément

### 10.1 Apprendre un vocabulaire et l’utiliser sont deux opérations différentes

L’appel `adapt()` de l’étape précédente a appris le vocabulaire sur l’entraînement. Cette opération est terminée et ne doit plus être répétée. Appeler maintenant `vectoriseur(...)` applique seulement la correspondance déjà figée :

$$
\phi(w)=
\begin{cases}
\operatorname{id}_{\mathrm{train}}(w), & w \in \mathcal{V}_{\mathrm{train}},\\
1, & w \notin \mathcal{V}_{\mathrm{train}}.
\end{cases}
$$

L’identifiant 1 représente `[UNK]`. L’identifiant 0 reste réservé à `<PAD>` et ne correspond à aucun vrai mot. Pour un poème $p$ contenant $n_p$ mots standardisés, nous obtenons :

$$
\tau(p)=(z_{p,1},z_{p,2},\ldots,z_{p,n_p}),
\qquad 1 \leq z_{p,j} < V,
$$

où $V=3203$ est la taille du vocabulaire.

Nous envoyons **un seul poème à la fois** au vectoriseur. Avec `ragged=True`, `flat_values` récupère uniquement ses vrais identifiants, sans ajouter de zéros pour l’aligner sur un autre poème. Ce lot unitaire sert au prétraitement ; il ne préjuge pas de la future taille des lots d’entraînement.

Nous transformons ici l’entraînement et la validation. Le test reste non vectorisé jusqu’à son évaluation finale.


In [13]:
def vectoriser_poemes_separement(liste_poemes):
    poemes_vectorises = []

    for poeme in liste_poemes:
        # Le crochet extérieur forme un lot contenant exactement un poème.
        resultat_ragged = vectoriseur(
            tf.constant([poeme["texte"]])
        )
        ids = resultat_ragged.flat_values.numpy().astype(
            np.int32,
            copy=False,
        )

        assert ids.ndim == 1
        assert len(ids) > 0
        assert np.all(ids != ID_PADDING)
        assert np.all(ids >= ID_HORS_VOCABULAIRE)
        assert np.all(ids < taille_vocabulaire)

        poemes_vectorises.append({
            "numero": poeme["numero"],
            "titre": poeme["titre"],
            "ids": ids,
        })

    return poemes_vectorises


vocabulaire_avant_vectorisation = list(vectoriseur.get_vocabulary())

poemes_ids_entrainement = vectoriser_poemes_separement(
    poemes_entrainement
)
poemes_ids_validation = vectoriser_poemes_separement(
    poemes_validation
)

assert vectoriseur.get_vocabulary() == vocabulaire_avant_vectorisation
assert sum(len(p["ids"]) for p in poemes_ids_entrainement) == 11242
assert sum(len(p["ids"]) for p in poemes_ids_validation) == 2529

print("Poèmes d’entraînement vectorisés :", len(poemes_ids_entrainement))
print("Poèmes de validation vectorisés   :", len(poemes_ids_validation))
print("Vocabulaire inchangé              : oui")
print("Jeu de test vectorisé             : non")


Poèmes d’entraînement vectorisés :

 34
Poèmes de validation vectorisés   : 7
Vocabulaire inchangé              : oui
Jeu de test vectorisé             : non


## 11. Construire les fenêtres « trois mots vers mot suivant »

### 11.1 Ajouter le padding uniquement au début de chaque poème

Pour conserver aussi les premiers mots, nous définissons une version étendue de la séquence :

$$
\widetilde z_{p,j}=
\begin{cases}
0, & j \leq 0,\\
z_{p,j}, & 1 \leq j \leq n_p.
\end{cases}
$$

Pour chaque position $t \in \{1,\ldots,n_p\}$, l’entrée et la cible sont :

$$
x_{p,t}=
(\widetilde z_{p,t-3},\widetilde z_{p,t-2},\widetilde z_{p,t-1}),
\qquad y_{p,t}=z_{p,t}.
$$

Les quatre premiers exemples du premier poème sont donc :

$$
(\text{PAD},\text{PAD},\text{PAD}) \rightarrow \text{mes},
$$

$$
(\text{PAD},\text{PAD},\text{mes}) \rightarrow \text{vers},
$$

$$
(\text{PAD},\text{mes},\text{vers}) \rightarrow \text{fuiraient},
$$

$$
(\text{mes},\text{vers},\text{fuiraient}) \rightarrow \text{mes}.
$$

Un poème de $n_p$ mots produit ainsi exactement $n_p$ exemples, et non $n_p-3$. Le padding est à gauche afin que le mot le plus récent soit toujours dans la dernière colonne. La future couche `Embedding(mask_zero=True)` pourra ignorer les zéros.

Le point essentiel est l’ordre des opérations : nous créons toutes les fenêtres **à l’intérieur d’un poème**, puis seulement nous concaténons les blocs.


In [14]:
TAILLE_CONTEXTE = 3


def creer_fenetres_un_poeme(ids, taille_contexte=TAILLE_CONTEXTE):
    ids = np.asarray(ids, dtype=np.int32)

    if ids.ndim != 1 or len(ids) == 0:
        raise ValueError("Le poème doit contenir au moins un identifiant.")
    if taille_contexte <= 0:
        raise ValueError("La taille du contexte doit être positive.")

    ids_avec_padding = np.pad(
        ids,
        (taille_contexte, 0),
        mode="constant",
        constant_values=ID_PADDING,
    )

    entrees = np.stack([
        ids_avec_padding[position:position + taille_contexte]
        for position in range(len(ids))
    ]).astype(np.int32, copy=False)
    cibles = ids.copy()

    return entrees, cibles


def assembler_fenetres(liste_poemes_vectorises):
    blocs_entrees = []
    blocs_cibles = []
    blocs_provenance = []
    statistiques = []

    for poeme in liste_poemes_vectorises:
        entrees, cibles = creer_fenetres_un_poeme(poeme["ids"])
        nombre_exemples = len(cibles)

        blocs_entrees.append(entrees)
        blocs_cibles.append(cibles)
        blocs_provenance.append(np.full(
            nombre_exemples,
            poeme["numero"],
            dtype=np.int32,
        ))
        statistiques.append({
            "numero": poeme["numero"],
            "titre": poeme["titre"],
            "mots": nombre_exemples,
            "fenetres": nombre_exemples,
        })

    return (
        np.concatenate(blocs_entrees, axis=0),
        np.concatenate(blocs_cibles, axis=0),
        np.concatenate(blocs_provenance, axis=0),
        statistiques,
    )


(
    X_entrainement,
    y_entrainement,
    provenance_entrainement,
    statistiques_fenetres_entrainement,
) = assembler_fenetres(poemes_ids_entrainement)

(
    X_validation,
    y_validation,
    provenance_validation,
    statistiques_fenetres_validation,
) = assembler_fenetres(poemes_ids_validation)


### 11.2 Prouver qu’aucune frontière n’est traversée

Nous conservons pour chaque fenêtre le numéro de son poème d’origine. Cette provenance sert uniquement à l’audit, jamais comme entrée du réseau. Le jeu $S$ est donc une union disjointe de blocs :

$$
\mathcal{D}_S
= \biguplus_{p \in \mathcal{P}_S}
\{(x_{p,t},y_{p,t},p) \mid 1 \leq t \leq n_p\}.
$$

Le nombre total d’exemples et les formes des tableaux sont :

$$
N_S=\sum_{p \in \mathcal{P}_S} n_p,
\qquad X_S \in \mathbb{N}_0^{N_S \times 3},
\qquad y_S \in \{1,\ldots,V-1\}^{N_S}.
$$

À chaque début de poème, la première entrée doit redevenir `[0, 0, 0]`. Nous reconstruisons également toutes les fenêtres attendues à partir des identifiants du poème et les comparons terme à terme aux tableaux finaux. Ce contrôle est plus fort qu’une simple vérification des dimensions.

La validation contient 488 cibles `[UNK]`. La proportion de cibles représentables par un mot connu vaut :

$$
1-\frac{488}{2529} \approx 80{,}70\%.
$$

Ainsi, un modèle qui ne prédit jamais `[UNK]` ne pourra pas dépasser environ 80,70 % d’accuracy globale sur cette validation, même s’il reconnaît parfaitement tous les mots connus. Plus tard, nous compléterons donc l’accuracy globale par une mesure sur les cibles connues.


In [15]:
def verifier_jeu_fenetres(
    liste_poemes_vectorises,
    entrees,
    cibles,
    provenance,
):
    position_globale = 0

    for poeme in liste_poemes_vectorises:
        ids = poeme["ids"]
        nombre_mots = len(ids)
        fin_bloc = position_globale + nombre_mots

        entrees_bloc = entrees[position_globale:fin_bloc]
        cibles_bloc = cibles[position_globale:fin_bloc]
        provenance_bloc = provenance[position_globale:fin_bloc]

        attendu_x, attendu_y = creer_fenetres_un_poeme(ids)
        assert np.array_equal(entrees_bloc, attendu_x)
        assert np.array_equal(cibles_bloc, attendu_y)
        assert np.all(provenance_bloc == poeme["numero"])
        assert np.array_equal(
            entrees_bloc[0],
            np.full(TAILLE_CONTEXTE, ID_PADDING, dtype=np.int32),
        )

        position_globale = fin_bloc

    assert position_globale == len(cibles)
    assert entrees.shape == (len(cibles), TAILLE_CONTEXTE)
    assert provenance.shape == cibles.shape
    assert np.all(cibles != ID_PADDING)
    assert np.all((entrees >= ID_PADDING) & (entrees < taille_vocabulaire))
    assert np.all(
        (cibles >= ID_HORS_VOCABULAIRE)
        & (cibles < taille_vocabulaire)
    )

    zeros_par_fenetre = np.sum(entrees == ID_PADDING, axis=1)
    nombre_poemes = len(liste_poemes_vectorises)
    assert np.count_nonzero(zeros_par_fenetre == 3) == nombre_poemes
    assert np.count_nonzero(zeros_par_fenetre == 2) == nombre_poemes
    assert np.count_nonzero(zeros_par_fenetre == 1) == nombre_poemes
    assert np.count_nonzero(entrees == ID_PADDING) == 6 * nombre_poemes


verifier_jeu_fenetres(
    poemes_ids_entrainement,
    X_entrainement,
    y_entrainement,
    provenance_entrainement,
)
verifier_jeu_fenetres(
    poemes_ids_validation,
    X_validation,
    y_validation,
    provenance_validation,
)

assert X_entrainement.shape == (11242, 3)
assert y_entrainement.shape == (11242,)
assert X_validation.shape == (2529, 3)
assert y_validation.shape == (2529,)
assert X_entrainement.dtype == y_entrainement.dtype == np.int32
assert X_validation.dtype == y_validation.dtype == np.int32
assert np.array_equal(y_entrainement, valeurs_train.astype(np.int32))
assert np.array_equal(y_validation, valeurs_validation.astype(np.int32))
assert np.count_nonzero(y_entrainement == ID_HORS_VOCABULAIRE) == 0
assert np.count_nonzero(y_validation == ID_HORS_VOCABULAIRE) == 488
assert vectoriseur.get_vocabulary() == vocabulaire_avant_vectorisation

def token_affichable(identifiant):
    if identifiant == ID_PADDING:
        return "<PAD>"
    return vocabulaire[int(identifiant)]


print("Formes finales :")
print("  entraînement :", X_entrainement.shape, y_entrainement.shape)
print("  validation   :", X_validation.shape, y_validation.shape)
print()
print("Quatre premières fenêtres d’entraînement :")
for indice in range(4):
    contexte = [
        token_affichable(id_token)
        for id_token in X_entrainement[indice]
    ]
    cible = token_affichable(y_entrainement[indice])
    print(f"  {contexte} -> {cible}")

premier_poeme = poemes_ids_entrainement[0]
deuxieme_poeme = poemes_ids_entrainement[1]
debut_deuxieme_bloc = len(premier_poeme["ids"])
fin_premier = [
    token_affichable(id_token)
    for id_token in premier_poeme["ids"][-3:]
]
debut_deuxieme = [
    token_affichable(id_token)
    for id_token in X_entrainement[debut_deuxieme_bloc]
]
cible_deuxieme = token_affichable(
    y_entrainement[debut_deuxieme_bloc]
)

print()
print("Audit d’une frontière :")
print(f"  fin du poème {premier_poeme['numero']} : {fin_premier}")
print(
    f"  début du poème {deuxieme_poeme['numero']} : "
    f"{debut_deuxieme} -> {cible_deuxieme}"
)
print("  La fin du premier poème n’entre pas dans la fenêtre suivante.")
print("  Le jeu de test est toujours non vectorisé.")


Formes finales :
  entraînement : (11242, 3) (11242,)
  validation   : (2529, 3) (2529,)

Quatre premières fenêtres d’entraînement :
  ['<PAD>', '<PAD>', '<PAD>'] -> mes
  ['<PAD>', '<PAD>', 'mes'] -> vers
  ['<PAD>', 'mes', 'vers'] -> fuiraient
  ['mes', 'vers', 'fuiraient'] -> mes

Audit d’une frontière :
  fin du poème 1 : ['ailes', 'comme', "l'amour"]
  début du poème 2 : ['<PAD>', '<PAD>', '<PAD>'] -> ce
  La fin du premier poème n’entre pas dans la fenêtre suivante.
  Le jeu de test est toujours non vectorisé.


## Bilan de l’étape 4

Chaque poème d’entraînement et de validation a été vectorisé séparément avec le vocabulaire figé. Le vectoriseur n’a pas été réadapté et le test reste scellé.

Le padding a ensuite été ajouté au début de chaque poème avant de construire les fenêtres. Nous obtenons :

- `X_entrainement` de forme **(11 242, 3)** et `y_entrainement` de forme **(11 242,)** ;
- `X_validation` de forme **(2 529, 3)** et `y_validation` de forme **(2 529,)** ;
- exactement une fenêtre entièrement remplie de zéros au début de chacun des 41 poèmes concernés ;
- aucune cible égale au padding et aucune fenêtre traversant deux poèmes.

Les 488 cibles `[UNK]` de validation sont conservées honnêtement. Elles représentent une limite importante pour l’accuracy globale et justifieront plus tard une mesure complémentaire sur les mots connus.

Aucun mélange ni lot d’entraînement n’a encore été créé. Les tableaux restent dans l’ordre des poèmes, avec une provenance séparée uniquement destinée aux contrôles.

**Prochaine étape :** créer les pipelines `tf.data.Dataset`, mélanger uniquement l’entraînement, choisir les lots, puis comprendre les couches `Embedding` et `SimpleRNN` avant de construire le réseau.


## 12. Transformer les tableaux en pipelines tf.data.Dataset

### 12.1 Un exemple, un lot et une époque

Nos tableaux NumPy sont déjà constitués de couples entrée-cible :

$$
\mathcal{D}_{\mathrm{train}}
=\{(x_i,y_i)\}_{i=1}^{N_{\mathrm{train}}},
\qquad N_{\mathrm{train}}=11\,242.
$$

Un élément individuel possède une entrée $x_i\in\mathbb{N}_0^3$ et une cible scalaire $y_i$. La méthode from_tensor_slices crée donc 11 242 éléments indépendants ; elle ne concatène ni les fenêtres ni les poèmes.

Nous regroupons ensuite au maximum $B=64$ exemples par lot. Pour un jeu de $N$ exemples, le nombre de lots est :

$$
K=\left\lceil\frac{N}{B}\right\rceil.
$$

Nous gardons le dernier lot même s’il est incomplet. Ainsi :

$$
\left\lceil\frac{11\,242}{64}\right\rceil=176
\quad\text{et}\quad
\left\lceil\frac{2\,529}{64}\right\rceil=40.
$$

Le dernier lot contient 42 exemples pour l’entraînement et 33 pour la validation. L’option drop_remainder=False empêche donc de perdre ces données.

### 12.2 Pourquoi mélanger uniquement l’entraînement ?

À l’époque $e$, le mélange applique une permutation $\pi_e$ aux indices d’entraînement :

$$
(x_1,y_1),\ldots,(x_N,y_N)
\longrightarrow
(x_{\pi_e(1)},y_{\pi_e(1)}),\ldots,(x_{\pi_e(N)},y_{\pi_e(N)}).
$$

Le couple $(x_i,y_i)$ reste intact : seule sa position change. Cela évite que les mises à jour successives du gradient suivent toujours l’ordre des poèmes. Comme chaque fenêtre contient déjà son contexte complet et que notre futur RNN ne sera pas stateful, deux exemples placés dans le même lot ne partagent aucun état caché.

La validation n’est pas mélangée. Son ordre stable facilite les contrôles et rend les comparaisons reproductibles. La graine fixe le processus aléatoire ; reshuffle_each_iteration=True produit une nouvelle permutation à chaque époque, mais la suite complète reste reproductible lorsque le notebook est relancé dans les mêmes conditions.

Enfin, prefetch prépare le lot suivant pendant que le lot courant est calculé. Il améliore l’alimentation du modèle sans modifier les exemples. Nous n’ajoutons ni repeat, car Keras gérera les époques, ni cache, car les tableaux sont déjà en mémoire.


In [16]:
TAILLE_LOT = 64


def creer_pipeline(entrees, cibles, *, melanger):
    if entrees.ndim != 2 or entrees.shape[1] != TAILLE_CONTEXTE:
        raise ValueError("Les entrées doivent avoir la forme (N, 3).")
    if cibles.ndim != 1 or len(entrees) != len(cibles):
        raise ValueError("Il faut exactement une cible par entrée.")
    if entrees.dtype != np.int32 or cibles.dtype != np.int32:
        raise TypeError("Les identifiants doivent être de type np.int32.")

    dataset = tf.data.Dataset.from_tensor_slices((entrees, cibles))

    if melanger:
        dataset = dataset.shuffle(
            buffer_size=len(entrees),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.batch(
        TAILLE_LOT,
        drop_remainder=False,
    )

    options = tf.data.Options()
    options.deterministic = True
    dataset = dataset.with_options(options)

    return dataset.prefetch(tf.data.AUTOTUNE)


dataset_entrainement = creer_pipeline(
    X_entrainement,
    y_entrainement,
    melanger=True,
)
dataset_validation = creer_pipeline(
    X_validation,
    y_validation,
    melanger=False,
)


### 12.3 Contrôler les pipelines avant le réseau

Après la mise en lots, les formes symboliques attendues sont :

$$
X_{\mathrm{lot}}\in\mathbb{N}_0^{B'\times3},
\qquad
y_{\mathrm{lot}}\in\mathbb{N}_0^{B'},
\qquad 1\leq B'\leq64.
$$

La dimension $B'$ vaut habituellement 64, mais reste notée None dans element_spec parce que le dernier lot peut être plus petit. Nous vérifions aussi que le premier lot de validation correspond exactement aux 64 premières lignes originales. Cela prouve qu’aucun mélange involontaire n’a été appliqué à la validation.

La provenance des poèmes n’entre pas dans ces pipelines : elle servait uniquement à l’audit des frontières. Le modèle reçoit seulement une matrice de trois identifiants et la cible associée.


In [17]:
NOMBRE_LOTS_ENTRAINEMENT = int(np.ceil(
    len(X_entrainement) / TAILLE_LOT
))
NOMBRE_LOTS_VALIDATION = int(np.ceil(
    len(X_validation) / TAILLE_LOT
))

assert int(dataset_entrainement.cardinality().numpy()) == 176
assert int(dataset_validation.cardinality().numpy()) == 40
assert NOMBRE_LOTS_ENTRAINEMENT == 176
assert NOMBRE_LOTS_VALIDATION == 40
assert len(X_entrainement) % TAILLE_LOT == 42
assert len(X_validation) % TAILLE_LOT == 33

spec_x_train, spec_y_train = dataset_entrainement.element_spec
assert spec_x_train.shape == (None, TAILLE_CONTEXTE)
assert spec_y_train.shape == (None,)
assert spec_x_train.dtype == tf.int32
assert spec_y_train.dtype == tf.int32

premier_x_train, premier_y_train = next(iter(dataset_entrainement))
premier_x_validation, premier_y_validation = next(
    iter(dataset_validation)
)

assert premier_x_train.shape == (TAILLE_LOT, TAILLE_CONTEXTE)
assert premier_y_train.shape == (TAILLE_LOT,)
assert premier_x_validation.shape == (
    TAILLE_LOT,
    TAILLE_CONTEXTE,
)
assert premier_y_validation.shape == (TAILLE_LOT,)
assert np.array_equal(
    premier_x_validation.numpy(),
    X_validation[:TAILLE_LOT],
)
assert np.array_equal(
    premier_y_validation.numpy(),
    y_validation[:TAILLE_LOT],
)

entrainement_effectivement_melange = not np.array_equal(
    premier_x_train.numpy(),
    X_entrainement[:TAILLE_LOT],
)

print("Avant les lots :")
print("  une entrée      :", X_entrainement[0].shape)
print("  une cible       : scalaire")
print("Après les lots :")
print("  spécification X :", spec_x_train)
print("  spécification y :", spec_y_train)
print()
print(
    "Entraînement :",
    NOMBRE_LOTS_ENTRAINEMENT,
    "lots ; dernier lot = 42 exemples",
)
print(
    "Validation   :",
    NOMBRE_LOTS_VALIDATION,
    "lots ; dernier lot = 33 exemples",
)
print(
    "Premier lot d’entraînement mélangé :",
    "oui" if entrainement_effectivement_melange else "non",
)
print("Validation restée dans son ordre : oui")
print("Jeu de test encore absent des pipelines : oui")


Avant les lots :


  une entrée      : (3,)
  une cible       : scalaire
Après les lots :
  spécification X : TensorSpec(shape=(None, 3), dtype=tf.int32, name=None)
  spécification y : TensorSpec(shape=(None,), dtype=tf.int32, name=None)

Entraînement : 176 lots ; dernier lot = 42 exemples
Validation   : 40 lots ; dernier lot = 33 exemples
Premier lot d’entraînement mélangé : oui
Validation restée dans son ordre : oui
Jeu de test encore absent des pipelines : oui


## 13. Comprendre la couche Embedding

### 13.1 Un identifiant n’est pas une quantité

Les nombres 85 et 124 sont seulement les adresses de deux mots dans le vocabulaire. Le fait que $124>85$ ne signifie pas que le second mot est plus grand ou plus important. Il serait donc incorrect de traiter directement les identifiants comme une variable numérique continue.

Une couche Embedding apprend une matrice contenant un vecteur par identifiant. Avec un vocabulaire de taille $V$ et une dimension d’embedding $D$ :

$$
E\in\mathbb{R}^{V\times D}.
$$

L’identifiant $i$ sélectionne simplement la ligne $i$ :

$$
\operatorname{Embedding}(i)=E[i,:]\in\mathbb{R}^{D}.
$$

Cette recherche de ligne, appelée lookup, est équivalente à la multiplication d’un vecteur one-hot $\mathbf{o}_i$ par la matrice :

$$
\mathbf{o}_i^{\top}E=E[i,:],
$$

mais elle évite de construire un grand vecteur presque entièrement nul. Les valeurs de $E$ sont initialisées aléatoirement, puis apprises par descente de gradient :

$$
E\leftarrow E-\eta\frac{\partial\mathcal{L}}{\partial E}.
$$

Des mots utiles dans des contextes voisins pourront ainsi recevoir des représentations proches. Ce n’est toutefois qu’un résultat possible de l’apprentissage, pas une propriété présente avant celui-ci.

### 13.2 Formes et nombre de paramètres

Pour $B$ fenêtres de trois mots, la transformation des formes est :

$$
(B,3)\longrightarrow(B,3,D).
$$

Les axes représentent successivement les exemples, les trois positions temporelles et les composantes du vecteur. Nous choisissons $D=64$. La couche contient alors :

$$
N_{\mathrm{Embedding}}=VD
=3\,203\times64
=204\,992
$$

paramètres entraînables et aucun biais.

### 13.3 Masquer le padding

Avec mask_zero=True, la couche produit aussi un masque booléen :

$$
m_{b,t}=\mathbf{1}[X_{b,t}\neq0].
$$

Par exemple, [0, 85, 124] produit le masque [False, True, True]. La future couche récurrente saura ainsi que 0 est un remplissage et non un vrai mot. En revanche, [UNK], d’identifiant 1, reste un véritable élément non masqué. Tous les mots inconnus partagent donc le même vecteur, ce qui explique pourquoi leur fréquence en validation devra être surveillée.


In [18]:
DIMENSION_EMBEDDING = 64
NOMBRE_UNITES_RNN = 64

ids_exemple = tf.constant([
    [ID_PADDING, ID_PADDING, ID_PADDING],
    [ID_PADDING, ID_PADDING, mot_vers_id["mes"]],
    [
        mot_vers_id["mes"],
        mot_vers_id["vers"],
        mot_vers_id["fuiraient"],
    ],
], dtype=tf.int32)

embedding_pedagogique = tf.keras.layers.Embedding(
    input_dim=taille_vocabulaire,
    output_dim=DIMENSION_EMBEDDING,
    mask_zero=True,
    name="embedding_pedagogique",
)

vecteurs_exemple = embedding_pedagogique(ids_exemple)
masque_exemple = embedding_pedagogique.compute_mask(ids_exemple)

assert ids_exemple.shape == (3, TAILLE_CONTEXTE)
assert vecteurs_exemple.shape == (
    3,
    TAILLE_CONTEXTE,
    DIMENSION_EMBEDDING,
)
assert masque_exemple.shape == (3, TAILLE_CONTEXTE)
assert embedding_pedagogique.count_params() == 204992
assert np.array_equal(
    masque_exemple.numpy(),
    ids_exemple.numpy() != ID_PADDING,
)

print("Identifiants :")
print(ids_exemple.numpy())
print("Masque (1 = vrai mot, 0 = padding) :")
print(masque_exemple.numpy().astype(np.int32))
print("Forme avant Embedding :", ids_exemple.shape)
print("Forme après Embedding :", vecteurs_exemple.shape)
print("Paramètres Embedding  :", embedding_pedagogique.count_params())


Identifiants :


[[  0   0   0]
 [  0   0  85]
 [ 85 124 950]]
Masque (1 = vrai mot, 0 = padding) :
[[0 0 0]
 [0 0 1]
 [1 1 1]]
Forme avant Embedding : (3, 3)
Forme après Embedding : (3, 3, 64)
Paramètres Embedding  : 204992


## 14. Comprendre la couche SimpleRNN

### 14.1 Lire trois vecteurs dans l’ordre

Après l’embedding, chaque position $t$ fournit un vecteur $\mathbf{e}_t\in\mathbb{R}^{D}$. Le RNN entretient un état caché $\mathbf{h}_t\in\mathbb{R}^{H}$ qui résume ce qu’il a déjà lu :

$$
\mathbf{h}_t
=\tanh\left(
W_x\mathbf{e}_t
+W_h\mathbf{h}_{t-1}
+\mathbf{b}
\right),
$$

avec :

$$
W_x\in\mathbb{R}^{H\times D},
\qquad
W_h\in\mathbb{R}^{H\times H},
\qquad
\mathbf{b}\in\mathbb{R}^{H}.
$$

Le terme $W_x\mathbf{e}_t$ apporte l’information du mot courant ; $W_h\mathbf{h}_{t-1}$ transmet le résumé précédent ; le biais déplace l’activation. La fonction $\tanh$ ajoute la non-linéarité et borne chaque composante entre -1 et 1. Keras stocke son noyau d’entrée avec la forme $(D,H)$ car il utilise des vecteurs-lignes ; c’est la même opération selon une autre convention d’écriture.

L’état initial est nul :

$$
\mathbf{h}_0=\mathbf{0}.
$$

Sur les trois positions, la même cellule et les mêmes poids sont réutilisés :

$$
\mathbf{h}_1=f(\mathbf{e}_1,\mathbf{h}_0),
\qquad
\mathbf{h}_2=f(\mathbf{e}_2,\mathbf{h}_1),
\qquad
\mathbf{h}_3=f(\mathbf{e}_3,\mathbf{h}_2).
$$

Avec return_sequences=False, seule la représentation finale $\mathbf{h}_3$ est renvoyée. L’ordre des mots compte donc. Le changement de formes est :

$$
(B,3,D)\longrightarrow(B,H).
$$

Notre réseau ne sera pas stateful : chaque fenêtre repart de $\mathbf{h}_0=\mathbf{0}$. Aucun état n’est transmis entre deux lignes d’un lot, entre deux lots ou entre deux poèmes.

### 14.2 Paramètres, masque et gradient

La couche apprend $DH$ poids d’entrée, $H^2$ poids récurrents et $H$ biais :

$$
N_{\mathrm{RNN}}=DH+H^2+H=H(D+H+1).
$$

Avec $D=64$ et $H=64$, cela donne :

$$
N_{\mathrm{RNN}}=64(64+64+1)=8\,256.
$$

Le masque fourni par l’embedding empêche un padding de mettre l’état à jour. Une écriture simplifiée est :

$$
\mathbf{h}_t
=m_t\,f(\mathbf{e}_t,\mathbf{h}_{t-1})
+(1-m_t)\mathbf{h}_{t-1}.
$$

Si $m_t=0$, alors $\mathbf{h}_t=\mathbf{h}_{t-1}$. Le contexte [PAD, PAD, mes] donne donc le même état final que la séquence contenant seulement mes. Pour [PAD, PAD, PAD], l’état reste nul ; la future couche de sortie devra alors apprendre, grâce à ses biais, quels mots commencent souvent un poème.

Pendant l’apprentissage, l’erreur sera propagée de $\mathbf{h}_3$ vers les états antérieurs : c’est la rétropropagation à travers le temps. Le Jacobien d’un pas contient :

$$
J_t
=\frac{\partial\mathbf{h}_t}{\partial\mathbf{h}_{t-1}}
=\operatorname{diag}(1-\mathbf{h}_t^2)W_h.
$$

Pour remonter plusieurs pas, ces matrices sont multipliées, par exemple $\partial\mathbf{h}_3/\partial\mathbf{h}_1=J_3J_2$. Les produits peuvent devenir très petits, disparition du gradient, ou très grands, explosion du gradient. Ici, le déroulement ne comporte que trois pas : le risque est bien plus faible que sur un long texte. Les GRU et LSTM deviennent surtout utiles lorsque l’on veut mémoriser des dépendances plus longues.


In [19]:
rnn_pedagogique = tf.keras.layers.SimpleRNN(
    units=NOMBRE_UNITES_RNN,
    activation="tanh",
    return_sequences=False,
    stateful=False,
    name="simple_rnn_pedagogique",
)

etats_exemple = rnn_pedagogique(
    vecteurs_exemple,
    mask=masque_exemple,
)

id_mes_seul = tf.constant([
    [mot_vers_id["mes"]],
], dtype=tf.int32)
vecteur_mes_seul = embedding_pedagogique(id_mes_seul)
masque_mes_seul = embedding_pedagogique.compute_mask(id_mes_seul)
etat_mes_seul = rnn_pedagogique(
    vecteur_mes_seul,
    mask=masque_mes_seul,
)

assert etats_exemple.shape == (3, NOMBRE_UNITES_RNN)
assert rnn_pedagogique.count_params() == 8256
np.testing.assert_allclose(
    etats_exemple[0].numpy(),
    np.zeros(NOMBRE_UNITES_RNN),
    atol=1e-7,
)
np.testing.assert_allclose(
    etats_exemple[1].numpy(),
    etat_mes_seul[0].numpy(),
    atol=1e-7,
)

print("Forme reçue par le RNN :", vecteurs_exemple.shape)
print("Forme renvoyée          :", etats_exemple.shape)
print("Paramètres SimpleRNN    :", rnn_pedagogique.count_params())
print("État du contexte tout-PAD nul : oui")
print("[PAD, PAD, mes] équivaut à [mes] : oui")
print("Ces couches pédagogiques ne sont pas encore entraînées.")


Forme reçue par le RNN : (3, 3, 64)
Forme renvoyée          : (3, 64)
Paramètres SimpleRNN    : 8256
État du contexte tout-PAD nul : oui
[PAD, PAD, mes] équivaut à [mes] : oui
Ces couches pédagogiques ne sont pas encore entraînées.


## Bilan de l’étape 5

Les fenêtres sont maintenant fournies à TensorFlow sous forme de pipelines :

- entraînement : 176 lots, mélangés à chaque époque de façon reproductible ;
- validation : 40 lots, conservés dans leur ordre ;
- taille maximale d’un lot : 64 ;
- aucun exemple supprimé et aucune répétition artificielle ;
- préchargement du lot suivant avec prefetch ;
- jeu de test toujours scellé.

Nous avons ensuite isolé les deux transformations fondamentales :

$$
(B,3)
\xrightarrow{\mathrm{Embedding}}
(B,3,64)
\xrightarrow{\mathrm{SimpleRNN}}
(B,64).
$$

L’embedding apprend 204 992 paramètres pour représenter les mots. Le SimpleRNN apprend 8 256 paramètres pour lire les trois vecteurs dans l’ordre. Les petites couches créées ici servent uniquement à vérifier les formes, le nombre de paramètres et le masque ; aucun apprentissage n’a encore eu lieu.

**Prochaine étape :** ajouter la couche Dense qui produit un score par mot du vocabulaire, comprendre les logits, le softmax et l’entropie croisée catégorielle creuse, puis construire et compiler le modèle complet.
